# Siamese Networks — Siamese Neural Networks for One-shot Image Recognition (2015)

_Papers · Meta-learning, Bayesian & Robustness_

**Train twin shared-weight networks to say "same or different?", then classify a brand-new class from a single example by picking the most similar support image.**

---

This notebook is a practice scaffold for the **Siamese Networks — Siamese Neural Networks for One-shot Image Recognition (2015)** lesson. We build it in stages — the similarity head, the tied encoder, the same/different training, and a one-shot evaluation that the encoder never saw during training. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — The similarity head from Section 3.1

The Siamese network compares two embeddings with a *weighted L1* distance fed through a sigmoid: `p = sigmoid(sum_j alpha_j |h1_j - h2_j|)`. The learned weights `alpha_j` decide how much each coordinate's gap matters. We first sanity-check this head by hand on a clearly-same pair and a clearly-different pair, so the formula is concrete before we wrap it in a model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from sklearn.datasets import load_digits

torch.manual_seed(0)
random.seed(0)

# Sanity-check the lesson's worked example: p = sigmoid(sum_j alpha_j |h1_j - h2_j|).
alpha = torch.tensor([-2.0, -1.0, -0.5])   # learned weights (negative here)

def head(h1, h2):                          # the paper's Section 3.1 similarity head
    gap = (h1 - h2).abs()                   # coordinate-wise |h1 - h2|
    z = (alpha * gap).sum()                 # weighted L1 distance
    return torch.sigmoid(z)

same = head(torch.tensor([0.80, 0.30, 0.55]), torch.tensor([0.78, 0.34, 0.50]))
diff = head(torch.tensor([0.9, 0.2, 0.8]), torch.tensor([0.1, 0.7, 0.1]))
print(f"worked example:  same-pair p = {same:.3f}   diff-pair p = {diff:.3f}")
# worked example:  same-pair p = 0.474   diff-pair p = 0.079

### Step 2 — Data split, the tied encoder, and the Siamese model

We use scikit-learn's 8x8 digits and split into a **train** set (for learning the verifier) and a **held-out** set used only for one-shot evaluation — so the encoder is judged on classes/images it tuned on versus images it never trained on. The key Siamese idea is *one* `Encoder` called twice: the weights are tied, so both twins are literally the same network. The `Siamese` wrapper then applies the weighted-L1 + sigmoid head from Step 1.

In [ ]:
# A tiny digit task: scikit-learn 8x8 digits (no download). Disjoint train / held-out images.
digits = load_digits()
imgs = torch.tensor(digits.images, dtype=torch.float32) / 16.0   # (1797, 8, 8) in [0, 1]
labels = torch.tensor(digits.target, dtype=torch.long)           # 10 classes (0-9)

N = len(labels)
perm = list(range(N))
random.Random(7).shuffle(perm)
cut = int(0.7 * N)
Xtr, ytr = imgs[perm[:cut]], labels[perm[:cut]]   # train the verifier on these
Xte, yte = imgs[perm[cut:]], labels[perm[cut:]]   # one-shot uses ONLY these held-out images

# ONE encoder = shared (tied) weights. Call it twice -> the twin networks.
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 2 * 2, 64)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # 8 -> 4
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # 4 -> 2
        return torch.sigmoid(self.fc(x.flatten(1)))  # embedding h in (0, 1)^64

# Siamese net: weighted-L1 distance + sigmoid (Section 3.1).
class Siamese(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()           # ONE encoder -> weights are tied
        self.alpha = nn.Linear(64, 1)  # the learned alpha_j (Section 3.1)
    def encode(self, x):
        return self.enc(x.unsqueeze(1))
    def forward(self, x1, x2):
        h1 = self.encode(x1)
        h2 = self.encode(x2)
        l1 = torch.abs(h1 - h2)                          # |h1 - h2|, coordinate-wise
        return torch.sigmoid(self.alpha(l1)).squeeze(1)  # p = sigmoid( sum_j alpha_j |h1_j - h2_j| )

### Step 3 — Pair sampling and the one-shot evaluator

Training needs balanced **same/different** pairs (Section 3.2): half drawn from one class (label 1), half from two different classes (label 0). Evaluation uses the **N-way one-shot** protocol (Section 4.3): show one support image per class, then pick the support whose similarity to the query is highest — `C* = argmax_c p(c)`. Both helpers are pure data-plumbing around the model we built in Step 2.

In [ ]:
def make_pairs(X, y, n, seed):   # balanced same/different pairs (Section 3.2)
    rng = random.Random(seed)
    by = {}
    for i, c in enumerate(y.tolist()):
        by.setdefault(c, []).append(i)
    cls = list(by.keys())
    a, b, lab = [], [], []
    for _ in range(n):
        if rng.random() < 0.5:            # SAME pair -> label 1
            c = rng.choice(cls)
            i, j = rng.sample(by[c], 2)
            t = 1.0
        else:                             # DIFFERENT pair -> label 0
            c1, c2 = rng.sample(cls, 2)
            i = rng.choice(by[c1])
            j = rng.choice(by[c2])
            t = 0.0
        a.append(i)
        b.append(j)
        lab.append(t)
    return X[a], X[b], torch.tensor(lab)

def oneshot(net, X, y, n_way, trials, seed=123):   # N-way one-shot: C* = argmax_c p(c) (Section 4.3)
    rng = random.Random(seed)
    by = {}
    for i, c in enumerate(y.tolist()):
        by.setdefault(c, []).append(i)
    classes = list(by.keys())
    net.eval()
    correct = 0
    with torch.no_grad():
        for _ in range(trials):
            ways = rng.sample(classes, n_way)
            true = rng.choice(ways)
            support = [rng.choice(by[c]) for c in ways]   # 1 example per class
            qi = rng.choice([k for k in by[true] if k not in support])
            q = X[qi].unsqueeze(0).repeat(n_way, 1, 1)
            s = X[support]
            if int(torch.argmax(net(q, s))) == ways.index(true):
                correct += 1
    return correct / trials

### Step 4 — Train the verifier and watch one-shot accuracy climb

We first record the **untrained** baseline: a random encoder, same one-shot eval — an honest ablation that isolates what training buys us. Then we train the same/different verifier with binary cross-entropy for 10 epochs, printing one-shot accuracy each epoch. Finally we report N-way one-shot on the held-out images the encoder never saw, for several N, against the `1/N` chance line.

In [ ]:
net = Siamese()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
bce = nn.BCELoss()

# ABLATION baseline: untrained (random) encoder, same one-shot eval.
print(f"untrained 5-way one-shot = {oneshot(net, Xte, yte, 5, 1000):.3f}   (chance = 1/5 = 0.200)")

# Train the verifier on same/different pairs, watch one-shot accuracy climb.
for ep in range(10):
    net.train()
    tot = 0.0
    for s in range(40):
        x1, x2, t = make_pairs(Xtr, ytr, 128, ep * 40 + s)
        opt.zero_grad()
        loss = bce(net(x1, x2), t)
        loss.backward()
        opt.step()
        tot += loss.item()
    print(f"  epoch {ep}  bce {tot/40:.4f}  5-way one-shot {oneshot(net, Xte, yte, 5, 1000):.3f}")

print("FINAL one-shot on HELD-OUT images (encoder never saw them):")
for nway in [2, 5, 10]:
    print(f"  {nway}-way one-shot = {oneshot(net, Xte, yte, nway, 2000):.3f}   chance = {1.0/nway:.3f}")
# Our small run: trained 5-way ~0.88 vs untrained ~0.40 and chance 0.20.
# Numbers vary by hardware/seed; this is our small run, not the paper's reported number.

## Visualize the data & results

_As we train the same/different verifier, does N-way one-shot accuracy on held-out images rise far above the 1/N chance baseline?_

### Step 1 — Rebuild the Siamese net and data self-contained

This panel re-runs the experiment on its own so it works even if you jump straight here. We rebuild the tied encoder, the weighted-L1 + sigmoid head, the train/held-out split, and the pair-sampling and one-shot helpers — identical logic to the reference cells above, just gathered in one place.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from sklearn.datasets import load_digits

torch.manual_seed(0)
random.seed(0)

digits = load_digits()
imgs = torch.tensor(digits.images, dtype=torch.float32) / 16.0
labels = torch.tensor(digits.target, dtype=torch.long)
N = len(labels)
perm = list(range(N))
random.Random(7).shuffle(perm)
cut = int(0.7 * N)
Xtr, ytr = imgs[perm[:cut]], labels[perm[:cut]]
Xte, yte = imgs[perm[cut:]], labels[perm[cut:]]   # one-shot uses ONLY held-out images

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 2 * 2, 64)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        return torch.sigmoid(self.fc(x.flatten(1)))

class Siamese(nn.Module):   # weighted-L1 + sigmoid head, tied encoder
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.alpha = nn.Linear(64, 1)
    def encode(self, x):
        return self.enc(x.unsqueeze(1))
    def forward(self, x1, x2):
        h1 = self.encode(x1)
        h2 = self.encode(x2)
        return torch.sigmoid(self.alpha(torch.abs(h1 - h2))).squeeze(1)

def make_pairs(X, y, n, seed):
    rng = random.Random(seed)
    by = {}
    for i, c in enumerate(y.tolist()):
        by.setdefault(c, []).append(i)
    cls = list(by.keys())
    a, b, lab = [], [], []
    for _ in range(n):
        if rng.random() < 0.5:
            c = rng.choice(cls)
            i, j = rng.sample(by[c], 2)
            t = 1.0
        else:
            c1, c2 = rng.sample(cls, 2)
            i = rng.choice(by[c1])
            j = rng.choice(by[c2])
            t = 0.0
        a.append(i)
        b.append(j)
        lab.append(t)
    return X[a], X[b], torch.tensor(lab)

def oneshot(net, X, y, n_way, trials, seed=123):
    rng = random.Random(seed)
    by = {}
    for i, c in enumerate(y.tolist()):
        by.setdefault(c, []).append(i)
    classes = list(by.keys())
    net.eval()
    correct = 0
    with torch.no_grad():
        for _ in range(trials):
            ways = rng.sample(classes, n_way)
            true = rng.choice(ways)
            support = [rng.choice(by[c]) for c in ways]
            qi = rng.choice([k for k in by[true] if k not in support])
            q = X[qi].unsqueeze(0).repeat(n_way, 1, 1)
            s = X[support]
            if int(torch.argmax(net(q, s))) == ways.index(true):
                correct += 1
    return correct / trials

### Step 2 — Train and trace one-shot accuracy per epoch

We print the untrained ablation baseline, then train the verifier and collect 5-way one-shot accuracy after every epoch — the trace should climb well above the `1/5 = 0.20` floor. The final line reports the harder 10-way one-shot, comparing it to its `0.10` chance level.

In [ ]:
net = Siamese()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
bce = nn.BCELoss()

print("untrained 5-way one-shot:", round(oneshot(net, Xte, yte, 5, 1000), 3))   # ~0.396 (ablation)

acc = []
for ep in range(10):
    net.train()
    for s in range(40):
        x1, x2, t = make_pairs(Xtr, ytr, 128, ep * 40 + s)
        opt.zero_grad()
        bce(net(x1, x2), t).backward()
        opt.step()
    acc.append(round(oneshot(net, Xte, yte, 5, 1000), 3))

print("trained 5-way one-shot per epoch:", acc)   # rises 0.57 -> 0.88
print("final 10-way one-shot:", round(oneshot(net, Xte, yte, 10, 2000), 3))   # ~0.78 vs chance 0.10
# Our small run, not the paper's reported number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a trained Siamese net that does 5-way one-shot well above the
            $1/5 = 20\%$ chance level. Now evaluate an untrained encoder &mdash; same architecture,
            random weights, no training &mdash; with the identical one-shot procedure. What accuracy do you
            expect, and what does the gap between trained and untrained demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep the architecture, support-set protocol, query images, and trial count identical; only skip the training loop (random initial weights). — _An honest ablation changes exactly one thing &mdash; whether the encoder was trained &mdash; so any difference is attributable to training the verification metric._
- Run the same N-way one-shot eval and read the accuracy. Expect it near &mdash; or only modestly above &mdash; the $1/N$ chance line. — _Random features carry little class structure, so nearest-by-similarity is close to guessing; any small lift is incidental separability of the toy images._
- Compare to the trained net's accuracy and attribute the gap to the learned similarity metric. — _Same architecture and protocol, so the only cause of the much higher trained accuracy is that binary-cross-entropy training shaped the embedding + $\alpha_j$ into a useful metric._

**Answer:** The untrained encoder sits near chance (in our run about 0.40 for 5-way, vs the
                 $0.20$ floor), while the trained net reaches roughly 0.88. Since architecture and
                 protocol are identical, the large gap isolates training the verification metric as the
                 cause &mdash; not the architecture, the support-set trick, or the data. The CODEVIZ panel shows
                 this contrast (our small run, not the paper's numbers).

</details>

**Problem 2.** Plug numbers through the head. Two embeddings are $h_1 = [0.6,\,0.1,\,0.4]$ and
            $h_2 = [0.5,\,0.3,\,0.4]$, with learned weights $\alpha = [-2.0,\,-1.0,\,-0.5]$. Compute the
            similarity $p = \sigma(\sum_j \alpha_j |h_{1,j} - h_{2,j}|)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Coordinate gaps: $|h_1 - h_2| = [\,|0.6-0.5|,\,|0.1-0.3|,\,|0.4-0.4|\,] = [0.1,\,0.2,\,0.0]$. — _The L1 head works coordinate-by-coordinate on absolute differences._
- Weighted sum: $z = (-2.0)(0.1) + (-1.0)(0.2) + (-0.5)(0.0) = -0.2 - 0.2 - 0 = -0.4$. — _Multiply each gap by its learned $\alpha_j$ and add, exactly as in Section 3.1._
- Sigmoid: $p = \sigma(-0.4) = 1/(1 + e^{0.4}) \approx 0.401$. — _The sigmoid maps the weighted distance to a probability in $(0,1)$._

**Answer:** $|h_1 - h_2| = [0.1, 0.2, 0.0]$, weighted sum $z = -0.4$, so
                 $p = \sigma(-0.4) \approx \mathbf{0.401}$. A moderate distance gives a middling similarity,
                 between the close-pair and far-pair cases of the worked example.

</details>

**Problem 3.** In a one-shot trial the test image scores these similarities against the four support images
            (4-way): $p = [0.31,\, 0.77,\, 0.62,\, 0.20]$ for classes $A, B, C, D$. Which class does the
            model predict, and what is chance accuracy here? If the true class is $B$, is the trial correct?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the decision rule $C^{*} = \arg\max_c p(c)$: the largest score is $0.77$ at class $B$. — _One-shot prediction is the support class with the highest similarity &mdash; not a threshold, just the max._
- Chance for 4-way one-shot is $1/4 = 25\%$. — _With one example per class and $N=4$ classes, random guessing is $1/N$._
- Compare prediction ($B$) to the true class ($B$). — _Accuracy counts a trial correct only when the $\arg\max$ class equals the true class._

**Answer:** The model predicts class $B$ (highest similarity, $0.77$). Chance accuracy is
                 $1/4 = 25\%$. Since the true class is $B$, the trial is correct. Note the prediction
                 uses the $\arg\max$ of $p$, so the absolute values do not need to exceed any threshold &mdash;
                 only their ranking matters.

</details>